# Blender Cycles + Tesla T4 Benchmark\n\nPrueba controlada: renderiza **un solo frame** del proyecto Neon Core con Cycles e intenta usar **OptiX primero y CUDA como fallback**.\n\nNo lanza la animación completa. Si el frame tarda más de 4 minutos, se cancela solo. Al final calcula una estimación lineal del tiempo total y guarda el benchmark en Drive.\n\nUso: activa **GPU T4** y pulsa **Ejecutar todo**.\n

In [ ]:
# @title Cycles T4 Benchmark — ejecutar todo
# Prueba segura: renderiza UN solo frame representativo con Cycles GPU.
# No lanza la animacion completa.

VERSION_MARKER = "CYCLES-T4-V1-2026-10-02"
PROJECT_SLUG = "blender-neon-core-demo"
TEST_FRAME = 74
CYCLES_SAMPLES = 32
MAX_TEST_SECONDS = 240

REPO_ZIP = "https://github.com/izquierdodavid528-code/Chat-gpt-work-/archive/refs/heads/main.zip"
REPO_DIR = "/content/chatgpt-workspace"
BLENDER_VERSION = "4.5.14"
BLENDER_DIR = f"/content/blender-{BLENDER_VERSION}-linux-x64"
BLENDER_BIN = f"{BLENDER_DIR}/blender"

import os
os.chdir("/content")

import json
import time
import shutil
import pathlib
import subprocess
import threading
import urllib.request
import zipfile
import tarfile
from datetime import datetime, timezone

def section(title):
    print("\n" + "="*78)
    print(title)
    print("="*78)

def run(cmd, check=True, capture=False, timeout=None):
    p = subprocess.run(
        ["bash", "-lc", cmd],
        cwd="/content",
        text=True,
        capture_output=capture,
        timeout=timeout,
    )
    if capture:
        if p.stdout:
            print(p.stdout)
        if p.stderr:
            print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Comando fallo ({p.returncode}): {cmd}")
    return p

section(VERSION_MARKER)

section("1/7 - Verificando Tesla T4")
gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    cwd="/content",
    text=True,
    capture_output=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError(
        "No hay GPU NVIDIA activa. Selecciona GPU T4 en Colab y vuelve a Ejecutar todo."
    )
print("GPU:", gpu.stdout.strip())

section("2/7 - Montando Drive")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/Remotion Projects")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive OK:", DRIVE_ROOT)

section("3/7 - Descargando workspace")
workspace_zip = pathlib.Path("/content/workspace-main.zip")
extract_root = pathlib.Path("/content/workspace-extract")

for p in [pathlib.Path(REPO_DIR), extract_root]:
    if p.exists():
        shutil.rmtree(p)
workspace_zip.unlink(missing_ok=True)

urllib.request.urlretrieve(REPO_ZIP, workspace_zip)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(workspace_zip, "r") as z:
    names = z.namelist()
    if not names:
        raise RuntimeError("ZIP del repositorio vacio.")
    top = names[0].split("/")[0]
    z.extractall(extract_root)

src_root = extract_root / top
shutil.move(str(src_root), REPO_DIR)
shutil.rmtree(extract_root, ignore_errors=True)
workspace_zip.unlink(missing_ok=True)
print("Workspace OK:", REPO_DIR)

section("4/7 - Instalando Blender")
run(
    "apt-get -qq update && "
    "apt-get -qq install -y curl ca-certificates ffmpeg "
    "libegl1 libgl1 libx11-6 libxi6 libxrender1 libxfixes3 "
    "libxxf86vm1 libxkbcommon0"
)

archive = pathlib.Path("/content/blender.tar.xz")
archive.unlink(missing_ok=True)

if not pathlib.Path(BLENDER_BIN).exists():
    filename = f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
    urls = [
        f"https://download.blender.org/release/Blender4.5/{filename}",
        f"https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.5/{filename}",
    ]

    ok = False
    for i, url in enumerate(urls, 1):
        print(f"Descarga Blender intento {i}/{len(urls)}:", url)
        q = subprocess.run(
            [
                "curl", "-fL",
                "--retry", "4",
                "--retry-delay", "2",
                "--connect-timeout", "30",
                "--max-time", "900",
                "-A", "Mozilla/5.0",
                url,
                "-o", str(archive),
            ],
            cwd="/content",
        )
        if q.returncode == 0 and archive.exists() and archive.stat().st_size > 300_000_000:
            ok = True
            print(f"Descarga OK: {archive.stat().st_size/1024/1024:.1f} MB")
            break
        archive.unlink(missing_ok=True)

    if not ok:
        raise RuntimeError("No se pudo descargar Blender desde ninguno de los origenes.")

    with tarfile.open(archive, "r:xz") as tf:
        tf.extractall("/content")
    archive.unlink(missing_ok=True)

ver = subprocess.run(
    [BLENDER_BIN, "--version"],
    cwd="/content",
    text=True,
    capture_output=True,
)
if ver.returncode != 0:
    raise RuntimeError("Blender no pudo iniciarse.")
print("\n".join(ver.stdout.splitlines()[:3]))

section("5/7 - Preparando escena Cycles GPU")
project_dir = pathlib.Path(REPO_DIR) / "projects" / PROJECT_SLUG
config_path = project_dir / "project.config.json"
scene_path = project_dir / "scene.py"

cfg = json.load(open(config_path, encoding="utf-8"))
source = scene_path.read_text(encoding="utf-8")

marker = "# Render tuning"
if marker not in source:
    raise RuntimeError("No se encontro el punto de corte esperado en scene.py.")

prefix = source.split(marker, 1)[0]

bench_script = project_dir / "_colab_cycles_benchmark.py"
bench_png = project_dir / "out" / "cycles-t4-frame.png"
bench_log = project_dir / "out" / "cycles-t4-benchmark.log"
gpu_log = project_dir / "out" / "cycles-t4-gpu.csv"
(project_dir / "out").mkdir(parents=True, exist_ok=True)

addon = f"""
# --- Colab Cycles T4 benchmark ---
scene.render.engine = "CYCLES"
scene.render.resolution_percentage = 100
scene.render.image_settings.file_format = "PNG"
scene.render.filepath = r"{bench_png}"
scene.frame_set({TEST_FRAME})

scene.cycles.samples = {CYCLES_SAMPLES}
try:
    scene.cycles.use_adaptive_sampling = True
    scene.cycles.adaptive_threshold = 0.01
except Exception:
    pass
try:
    scene.cycles.use_denoising = True
except Exception:
    pass

prefs = bpy.context.preferences.addons["cycles"].preferences
selected_backend = None
enabled_devices = []

for backend in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = backend
        try:
            prefs.refresh_devices()
        except Exception:
            try:
                prefs.get_devices()
            except Exception:
                pass

        devices = list(prefs.devices)
        gpu_devices = [d for d in devices if getattr(d, "type", "") != "CPU"]
        if gpu_devices:
            for d in devices:
                d.use = getattr(d, "type", "") != "CPU"
            selected_backend = backend
            enabled_devices = [
                {{"name": getattr(d, "name", ""), "type": getattr(d, "type", ""), "use": bool(getattr(d, "use", False))}}
                for d in devices
            ]
            break
    except Exception as e:
        print("BACKEND_FAIL", backend, repr(e))

if not selected_backend:
    raise RuntimeError("Cycles no encontro dispositivo GPU OPTIX/CUDA.")

scene.cycles.device = "GPU"
print("CYCLES_BACKEND", selected_backend)
print("CYCLES_DEVICES", enabled_devices)
print("CYCLES_SAMPLES", scene.cycles.samples)
print("CYCLES_FRAME", scene.frame_current)
print("CYCLES_RESOLUTION", scene.render.resolution_x, scene.render.resolution_y, scene.render.resolution_percentage)

bpy.ops.render.render(write_still=True)
print("CYCLES_T4_BENCHMARK_DONE")
"""

bench_script.write_text(prefix + "\n" + addon, encoding="utf-8")
print("Proyecto:", PROJECT_SLUG)
print("Frame de prueba:", TEST_FRAME)
print("Samples:", CYCLES_SAMPLES)
print("Resolucion:", cfg.get("resolution"))

section("6/7 - Renderizando UN frame con Cycles GPU")
stop = threading.Event()
samples = []

def poll_gpu():
    with open(gpu_log, "w", encoding="utf-8") as f:
        f.write("timestamp,gpu_util_percent,memory_used_mb\n")
        while not stop.is_set():
            p = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,memory.used",
                    "--format=csv,noheader,nounits",
                ],
                cwd="/content",
                text=True,
                capture_output=True,
            )
            if p.returncode == 0 and p.stdout.strip():
                vals = [x.strip() for x in p.stdout.strip().splitlines()[0].split(",")]
                try:
                    samples.append((float(vals[0]), float(vals[1])))
                except Exception:
                    pass
                f.write(f"{datetime.now(timezone.utc).isoformat()},{','.join(vals)}\n")
                f.flush()
            stop.wait(2)

t = threading.Thread(target=poll_gpu, daemon=True)
t.start()

cmd = [BLENDER_BIN, "-b", "--python", str(bench_script)]
print("Comando:", " ".join(cmd))
started = time.time()

with open(bench_log, "w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        cmd,
        cwd="/content",
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

    while proc.poll() is None:
        elapsed = time.time() - started
        if elapsed > MAX_TEST_SECONDS:
            proc.terminate()
            try:
                proc.wait(15)
            except Exception:
                proc.kill()
            stop.set()
            t.join(timeout=5)
            raise RuntimeError(
                f"El frame supero {MAX_TEST_SECONDS}s. Se cancelo automaticamente."
            )
        if samples:
            util, mem = samples[-1]
            print(f"{elapsed:.0f}s | GPU {util:.0f}% | VRAM {mem:.0f} MB")
        else:
            print(f"{elapsed:.0f}s | esperando lectura GPU...")
        time.sleep(10)

elapsed = time.time() - started
stop.set()
t.join(timeout=5)

if proc.returncode != 0:
    tail = "\n".join(open(bench_log, errors="ignore").read().splitlines()[-120:])
    print("\nULTIMAS LINEAS DEL LOG:\n" + tail)
    raise RuntimeError("El benchmark de Cycles fallo.")

print(f"FRAME COMPLETADO EN {elapsed:.1f}s")

section("7/7 - Resultado y estimacion")
utils = [x[0] for x in samples]
mems = [x[1] for x in samples]

estimate_seconds = elapsed * int(cfg.get("frames", 120))
estimate_minutes = estimate_seconds / 60

log_text = bench_log.read_text(encoding="utf-8", errors="ignore")
backend = None
for line in log_text.splitlines():
    if line.startswith("CYCLES_BACKEND "):
        backend = line.split(" ", 1)[1].strip()
        break

summary = {
    "version_marker": VERSION_MARKER,
    "project": PROJECT_SLUG,
    "frame": TEST_FRAME,
    "cycles_samples": CYCLES_SAMPLES,
    "backend": backend,
    "gpu": gpu.stdout.strip(),
    "frame_seconds": round(elapsed, 1),
    "estimated_full_animation_minutes": round(estimate_minutes, 1),
    "gpu_util_avg_percent": round(sum(utils)/len(utils), 1) if utils else None,
    "gpu_util_max_percent": round(max(utils), 1) if utils else None,
    "gpu_memory_max_mb": round(max(mems), 1) if mems else None,
    "completed_at": datetime.now(timezone.utc).isoformat(),
}

summary_path = project_dir / "out" / "cycles-t4-benchmark.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

drive_project = cfg.get("driveProjectDir", PROJECT_SLUG)
drive_out = DRIVE_ROOT / drive_project / "cycles-t4-benchmark"
drive_out.mkdir(parents=True, exist_ok=True)

for f in [bench_png, bench_log, gpu_log, summary_path]:
    if f.exists():
        shutil.copy2(f, drive_out / f.name)

print("\nLISTO")
print("Resultados guardados en:", drive_out)
print(f"1 frame: {elapsed:.1f}s")
print(f"Estimacion lineal para {cfg.get('frames', 120)} frames: {estimate_minutes:.1f} min")
print("Uso GPU medio:", summary["gpu_util_avg_percent"], "%")
print("Uso GPU maximo:", summary["gpu_util_max_percent"], "%")
